# AADP Colab runner - LANE D (AADP_exchange_d)

Control-plane notebook for cloud training. Attach this notebook to a Colab GPU runtime
via the VS Code Colab extension and keep the tab active while the agent drives it.

Cell order per session: `[probe]` → `[mount]` → `[bootstrap]` → `[agent]`. `[agent]` runs
the daemon synchronously and **stays busy for the whole session — that is the keepalive**
(Colab's idle timer only counts executing cells; background GPU work does not). After it
starts, the local CLI takes over (`python colab/cloud_sync.py cmd/hb/unassign`) — no more
Quick Picks. The remaining cells (`[args]` → `[launch]` → `[poll]` → `[collect]`) are the
manual fallback when the daemon is down (stop it first via `@stop` or interrupting
`[agent]`). The agent edits **only** the `[args]` cell.
Protocol and failure modes: `colab/COLAB.md`.

In [ ]:
# [probe] runtime facts - run first every session
import os, shutil, subprocess, sys
print("python:", sys.version.split()[0])
nv = shutil.which("nvidia-smi")
print("gpu:", subprocess.run([nv, "-L"], capture_output=True, text=True).stdout.strip() if nv else "none")
print(f"disk_free_gb: {shutil.disk_usage('/').free / 1e9:.1f}")
print("is_colab_vm:", os.path.exists("/content"))
try:
    import torch
    print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
except ImportError:
    print("torch: missing")

In [ ]:
# [mount] Google Drive - one OAuth click per runtime
import os
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")
os.environ["AADP_EXCHANGE_DIR"] = "AADP_exchange_d"  # LANE D - do not run against the main exchange
LANE = os.environ["AADP_EXCHANGE_DIR"]
EXCHANGE = Path("/content/drive/MyDrive") / LANE
EXCHANGE.mkdir(exist_ok=True)
print("exchange:", EXCHANGE, "| code_bundles:", len(list((EXCHANGE / 'code').glob('code_*.tar.gz'))) if (EXCHANGE / 'code').exists() else 0)

In [ ]:
# [bootstrap] newest code bundle + data + deps; snapshots baseline for [collect]
import csv, json, os, shutil, subprocess, sys, tarfile, time
from pathlib import Path

EXCHANGE = Path("/content/drive/MyDrive") / os.environ.get("AADP_EXCHANGE_DIR", "AADP_exchange")
WORK = Path("/content/AADP")
KEEP = {"open", "experiments", "logs"}

bundles = sorted((EXCHANGE / "code").glob("code_*.tar.gz"))
assert bundles, "no code bundle on Drive; run locally: python colab/cloud_sync.py push"
bundle = bundles[-1]
WORK.mkdir(exist_ok=True)
for item in WORK.iterdir():
    if item.name not in KEEP:
        shutil.rmtree(item) if item.is_dir() else item.unlink()
with tarfile.open(bundle) as tf:
    tf.extractall(WORK)
manifest = json.loads((WORK / "cloud_manifest.json").read_text())
print("bundle:", bundle.name)
print("commit:", manifest["commit"], "| dirty_files:", manifest["dirty_file_count"])

if not (WORK / "open/data/train.jsonl").exists():
    with tarfile.open(EXCHANGE / "data/open_data.tar.gz") as tf:
        tf.extractall(WORK)
print("train_jsonl_mb:", round((WORK / "open/data/train.jsonl").stat().st_size / 1e6, 1))

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers==4.46.3", "sentencepiece", "safetensors"], check=True)
import torch, transformers
print("torch:", torch.__version__, "| transformers:", transformers.__version__, "| cuda:", torch.cuda.is_available())

with (WORK / "experiments/results.csv").open(newline="", encoding="utf-8") as f:
    baseline_ids = [row["experiment_id"] for row in csv.DictReader(f)]
Path("/content/aadp_state.json").write_text(json.dumps(
    {"baseline_experiment_ids": baseline_ids, "bootstrap_ts": time.time(), "commit": manifest["commit"]}))
(WORK / "logs").mkdir(exist_ok=True)
print("baseline_rows:", len(baseline_ids))

In [ ]:
# [agent] run the Drive command-channel daemon SYNCHRONOUSLY - the session's last Quick Pick.
# The busy cell IS the keepalive: Colab's idle timer only counts executing cells, so a
# detached daemon leaves the kernel idle and the runtime gets reclaimed ~90 min after the
# last cell run (2026-07-03 incident). This cell runs until @stop / @unassign / interrupt.
# On daemon exit code 86 (@unassign or idle limit) this cell releases the runtime -- the
# daemon itself cannot: google.colab runtime.unassign() only works from the kernel.
# After it starts, the local side drives everything: python colab/cloud_sync.py cmd/hb/unassign
import json, os, subprocess, sys, time
from pathlib import Path

WORK = Path("/content/AADP")
logs = WORK / "logs"
logs.mkdir(exist_ok=True)
state_path = logs / "agent_daemon.json"

os.environ.setdefault("AADP_IDLE_MAX_MIN", "45")

rc = None
prev = json.loads(state_path.read_text()) if state_path.exists() else None
if prev and Path(f"/proc/{prev['pid']}/stat").exists() \
        and Path(f"/proc/{prev['pid']}/stat").read_text().rsplit(")", 1)[1].split()[0] not in ("Z",):
    print("daemon already running, pid:", prev["pid"], "- send @stop first to restart it")
else:
    log_path = logs / "agent_daemon.log"
    proc = subprocess.Popen([sys.executable, "-u", str(WORK / "colab/vm_agent.py"), "daemon"],
                            cwd=WORK, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    state_path.write_text(json.dumps({"pid": proc.pid, "log": str(log_path),
                                      "started_utc": time.strftime("%Y%m%d_%H%M%S", time.gmtime())}))
    print("daemon pid:", proc.pid, "| synchronous: this cell staying busy is the keepalive")
    try:
        with log_path.open("a") as lf:
            for line in proc.stdout:
                print(line, end="")
                lf.write(line)
                lf.flush()
    except KeyboardInterrupt:
        proc.terminate()
    finally:
        rc = proc.wait()
        print("daemon exited rc =", rc)

if rc == 86:
    print("releasing runtime (kernel-side unassign)")
    from google.colab import runtime
    runtime.unassign()

In [ ]:
# [args] the ONLY cell the agent edits per run; then run [launch]
RUN_SCRIPT = "train_transformer.py"
RUN_ARGS = (
    "--device cuda --base-model xlm-roberta-base --serializer state_v2 "
    "--max-length 384 --epochs 5 --batch-size 16 --eval-batch-size 64 --lr 2e-5 "
    "--class-weight-power 0.5 --label-smoothing 0.02 "
    "--replay-mode last1 --max-replay-samples 10000 --replay-sample-weight 0.5 "
    "--keep-threshold 0.0 --tokenize-batch-size 1024 "
    "--experiment-suffix cloud_state_v2_len384_5ep_fixed "
    "--notes 'G1 judgment run: state_v2 len384 5ep replay_last1 fixed-session; compare raw vs current_v1 len192 5ep fixed raw 0.739664 (+0.006 gate)' "
    "--no-research-log"
)
print(RUN_SCRIPT, RUN_ARGS)

In [ ]:
# [launch] background start; never run training synchronously in a cell
import json, shlex, subprocess, sys, time
from pathlib import Path

WORK = Path("/content/AADP")
logs = WORK / "logs"
logs.mkdir(exist_ok=True)
stamp = time.strftime("%Y%m%d_%H%M%S", time.gmtime())
log_path = logs / f"run_{stamp}.log"
cmd = [sys.executable, "-u", RUN_SCRIPT] + shlex.split(RUN_ARGS)
proc = subprocess.Popen(cmd, cwd=WORK, stdout=log_path.open("w"), stderr=subprocess.STDOUT, start_new_session=True)
(logs / "last_run.json").write_text(json.dumps(
    {"pid": proc.pid, "log": str(log_path), "script": RUN_SCRIPT, "args": RUN_ARGS, "started_utc": stamp}, indent=2))
print("pid:", proc.pid, "| log:", log_path.name)

In [24]:
# [poll] cheap status; rerun as needed (survives kernel reconnects)
# note: a bare /proc/<pid> check reports zombies as alive; read the stat state instead
import json, os, shutil, subprocess
from pathlib import Path

run = json.loads(Path("/content/AADP/logs/last_run.json").read_text())
pid = run["pid"]
try:
    os.waitpid(pid, os.WNOHANG)  # reap if it exited and the kernel is its parent
except ChildProcessError:
    pass
stat = Path(f"/proc/{pid}/stat")
state = stat.read_text().rsplit(")", 1)[1].split()[0] if stat.exists() else "gone"
alive = state not in ("Z", "gone")
print("pid:", pid, "| state:", state, "| alive:", alive, "| log:", run["log"])
print(subprocess.run(["tail", "-n", "15", run["log"]], capture_output=True, text=True).stdout)
if alive and shutil.which("nvidia-smi"):
    print("gpu:", subprocess.run(["nvidia-smi", "--query-gpu=utilization.gpu,memory.used",
                                  "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())

pid: 10555 | state: gone | alive: False | log: /content/AADP/logs/run_20260702_141656.log
    lint_or_typecheck  0.6239
    ask_user           0.6256
    glob_pattern       0.6334
    plan_task          0.6590
  top confusions:
    grep_search        -> read_file          488
    read_file          -> grep_search        342
    read_file          -> list_directory     307
    grep_search        -> list_directory     257
    list_directory     -> read_file          250
    glob_pattern       -> read_file          195
    glob_pattern       -> list_directory     134
    ask_user           -> plan_task          120
    glob_pattern       -> grep_search        116
    run_bash           -> run_tests          113



In [25]:
# [collect] ship new results rows + fresh artifacts to Drive:runs/<name>/
import csv, json, os, shutil, time
from pathlib import Path

WORK = Path("/content/AADP")
EXCHANGE = Path("/content/drive/MyDrive") / os.environ.get("AADP_EXCHANGE_DIR", "AADP_exchange")
EXTRA_PATHS = []  # e.g. ["model_out_final"] for final refit artifacts (large, slow copy)

state = json.loads(Path("/content/aadp_state.json").read_text())
base_ids = set(state["baseline_experiment_ids"])
with (WORK / "experiments/results.csv").open(newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    fieldnames = reader.fieldnames
    new_rows = [r for r in reader if r["experiment_id"] not in base_ids]
assert new_rows, "no new results rows since bootstrap"

stamp = time.strftime("%Y%m%d_%H%M%S", time.gmtime())
run_name = f"{stamp}_{new_rows[-1]['experiment_id'][:60]}"
out = EXCHANGE / "runs" / run_name
for sub in ("logits", "artifacts"):
    (out / sub).mkdir(parents=True, exist_ok=True)
with (out / "results_rows.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(new_rows)
copied = 0
for sub in ("logits", "artifacts"):
    src = WORK / "experiments" / sub
    if src.exists():
        for p in src.iterdir():
            if p.is_file() and p.stat().st_mtime >= state["bootstrap_ts"]:
                shutil.copy2(p, out / sub / p.name)
                copied += 1
for extra in EXTRA_PATHS:
    src = WORK / extra
    shutil.copytree(src, out / "extra" / src.name, dirs_exist_ok=True)
(out / "manifest.json").write_text(json.dumps(
    {"commit": state["commit"], "rows": len(new_rows), "files": copied, "created_utc": stamp}, indent=2))
print("run:", run_name, "| rows:", len(new_rows), "| files:", copied)
print("local next: .venv/bin/python colab/cloud_sync.py pull", run_name)

run: 20260702_161434_20260702_155500_gpu_transformer_session_state_v2_len384_repl | rows: 1 | files: 2
local next: .venv/bin/python colab/cloud_sync.py pull 20260702_161434_20260702_155500_gpu_transformer_session_state_v2_len384_repl


In [ ]:
import json, os, shutil, subprocess
from pathlib import Path

run = json.loads(Path("/content/AADP/logs/last_run.json").read_text())
pid = run["pid"]
try:
    os.waitpid(pid, os.WNOHANG)
except ChildProcessError:
    pass
stat = Path(f"/proc/{pid}/stat")
state = stat.read_text().rsplit(")", 1)[1].split()[0] if stat.exists() else "gone"
alive = state not in ("Z", "gone")
print("pid:", pid, "| state:", state, "| alive:", alive, "| log:", run["log"])
print(subprocess.run(["tail", "-n", "25", run["log"]], capture_output=True, text=True).stdout)
if alive and shutil.which("nvidia-smi"):
    print("gpu:", subprocess.run(["nvidia-smi", "--query-gpu=utilization.gpu,memory.used",
                                  "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())


In [ ]:
import json, os, shutil, subprocess
from pathlib import Path

run = json.loads(Path("/content/AADP/logs/last_run.json").read_text())
pid = run["pid"]
try:
    os.waitpid(pid, os.WNOHANG)
except ChildProcessError:
    pass
stat = Path(f"/proc/{pid}/stat")
state = stat.read_text().rsplit(")", 1)[1].split()[0] if stat.exists() else "gone"
alive = state not in ("Z", "gone")
print("pid:", pid, "| state:", state, "| alive:", alive, "| log:", run["log"])
print(subprocess.run(["tail", "-n", "25", run["log"]], capture_output=True, text=True).stdout)
if alive and shutil.which("nvidia-smi"):
    print("gpu:", subprocess.run(["nvidia-smi", "--query-gpu=utilization.gpu,memory.used",
                                  "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
